# 02: Classification — Logistic Regression to Gradient Boosting

## Module 2, Week 5 — Classification & Tree-Based Models

---

### Learning Objectives

By the end of this notebook you will be able to:

1. **Understand logistic regression** — how it uses the sigmoid function to produce probabilities for binary classification
2. **Evaluate classifiers properly** — using precision, recall, F1-score, confusion matrices, ROC-AUC, and PR-AUC (not just accuracy!)
3. **Build and interpret decision trees** — understand how they split data and why they tend to overfit
4. **Use ensemble methods** — Random Forest (bagging) and Gradient Boosting to get better, more robust predictions
5. **Compare XGBoost and LightGBM** — the industry-standard gradient boosting libraries
6. **Tune hyperparameters** — using GridSearchCV to find optimal model settings

### Prerequisites
- Module 2 Week 4 (01-regression-and-regularization) or equivalent knowledge of sklearn basics
- Basic understanding of train/test splits and cross-validation

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc,
                             precision_recall_curve, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

from shared.viz_utils import setup_style, save_fig
setup_style()

print("All imports successful!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
import sklearn
print(f"Scikit-learn: {sklearn.__version__}")

---

## Section 1: Logistic Regression

### What is Logistic Regression?

Despite the name, **logistic regression is a classification algorithm**, not a regression algorithm. The name comes from the fact that it uses a regression-like equation internally, but the output is a **probability** (between 0 and 1) that gets mapped to a class label.

### The Sigmoid Function

Logistic regression takes a linear combination of features and passes it through the **sigmoid function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

where $z = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$

The sigmoid function has these key properties:
- Output is always between 0 and 1 (perfect for probabilities!)
- When $z = 0$, output is 0.5 (the decision boundary)
- When $z$ is very large, output approaches 1
- When $z$ is very negative, output approaches 0

### Log-Odds (Logit)

The inverse of the sigmoid is the **logit function**:

$$\text{logit}(p) = \ln\left(\frac{p}{1-p}\right) = w_0 + w_1 x_1 + \dots + w_n x_n$$

The term $\frac{p}{1-p}$ is the **odds ratio** — the ratio of the probability of success to failure.

### Decision Boundary

By default, the decision boundary is at probability = 0.5:
- If $P(\text{class}=1) \geq 0.5$ → predict class 1
- If $P(\text{class}=1) < 0.5$ → predict class 0

You can adjust this threshold depending on your use case (more on this later).

### Cost Function: Binary Cross-Entropy

Logistic regression minimizes **binary cross-entropy** (also called log loss):

$$J = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i) + (1 - y_i)\log(1 - \hat{p}_i)\right]$$

This penalizes confident wrong predictions heavily — if you predict 0.99 for a sample that is actually class 0, the penalty is huge.

In [ ]:
# --- Visualize the Sigmoid Function ---
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(z, sigmoid, linewidth=2.5, color='steelblue')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='Decision boundary (p=0.5)')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('z (linear combination of features)', fontsize=12)
ax.set_ylabel('σ(z) = Probability', fontsize=12)
ax.set_title('The Sigmoid Function', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)

# Annotate regions
ax.annotate('Predict Class 0', xy=(-5, 0.15), fontsize=12, color='red', fontweight='bold')
ax.annotate('Predict Class 1', xy=(3, 0.85), fontsize=12, color='green', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Generate 2D Classification Data ---
np.random.seed(42)
X_2d, y_2d = make_classification(
    n_samples=300,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y_2d, test_size=0.3, random_state=42
)

print(f"Training set: {X_train_2d.shape[0]} samples")
print(f"Test set:     {X_test_2d.shape[0]} samples")
print(f"Class distribution (train): {np.bincount(y_train_2d)}")

# --- Fit Logistic Regression ---
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_2d, y_train_2d)

train_acc = log_reg.score(X_train_2d, y_train_2d)
test_acc = log_reg.score(X_test_2d, y_test_2d)

print(f"\nLogistic Regression Results:")
print(f"  Train accuracy: {train_acc:.4f}")
print(f"  Test accuracy:  {test_acc:.4f}")
print(f"  Coefficients:   {log_reg.coef_[0]}")
print(f"  Intercept:      {log_reg.intercept_[0]:.4f}")

In [ ]:
# --- Plot Decision Boundary for Logistic Regression ---

def plot_decision_boundary(model, X, y, ax=None, title="Decision Boundary", alpha=0.3):
    """Plot the decision boundary of a classifier on 2D data."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # Create meshgrid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                          np.arange(y_min, y_max, 0.02))

    # Predict on the meshgrid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot decision regions
    ax.contourf(xx, yy, Z, alpha=alpha, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5)

    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu',
                         edgecolors='black', linewidth=0.5, s=40)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Feature 1', fontsize=11)
    ax.set_ylabel('Feature 2', fontsize=11)
    return ax

fig, ax = plt.subplots(figsize=(9, 6))
plot_decision_boundary(log_reg, X_test_2d, y_test_2d, ax=ax,
                       title='Logistic Regression — Decision Boundary')
plt.tight_layout()
plt.show()

print("\nKey insight: Logistic regression creates a LINEAR decision boundary.")
print("It draws a straight line (or hyperplane in higher dimensions) to separate classes.")

---

## Section 2: Classification Metrics

### Why Accuracy Alone is Not Enough

Imagine you are building a **spam email detector**. Your dataset has:
- 900 legitimate emails (class 0)
- 100 spam emails (class 1)

A "dumb" model that **always predicts "not spam"** would get **90% accuracy** — sounds great, right? But it catches **zero** spam! This is why we need better metrics.

### The Confusion Matrix

A confusion matrix shows us exactly what our model gets right and wrong:

```
                    Predicted
                 Negative  Positive
Actual Negative [   TN    |   FP   ]    TN = True Negative  (correctly said "not spam")
Actual Positive [   FN    |   TP   ]    FP = False Positive (wrongly said "spam") — Type I Error
                                        FN = False Negative (missed spam)        — Type II Error
                                        TP = True Positive  (correctly caught spam)
```

### Key Metrics (using spam example)

| Metric | Formula | Meaning | Spam Example |
|--------|---------|---------|--------------|
| **Accuracy** | (TP + TN) / Total | Overall correctness | "What fraction of all emails did we classify correctly?" |
| **Precision** | TP / (TP + FP) | Quality of positive predictions | "Of all emails we flagged as spam, how many were actually spam?" |
| **Recall** (Sensitivity) | TP / (TP + FN) | Coverage of actual positives | "Of all actual spam, how much did we catch?" |
| **F1-Score** | 2 * (Precision * Recall) / (Precision + Recall) | Harmonic mean of P & R | Balances precision and recall |

### When to Prioritize Which?

- **High Precision needed**: When false positives are costly (e.g., marking a legitimate email as spam — user misses important email)
- **High Recall needed**: When false negatives are costly (e.g., missing a cancer diagnosis, missing fraud)
- **F1-Score**: When you need a balance of both

### ROC-AUC and PR-AUC

- **ROC Curve**: Plots True Positive Rate vs False Positive Rate at different thresholds. AUC = 1.0 is perfect, 0.5 is random guessing.
- **PR Curve** (Precision-Recall): More informative than ROC when classes are **imbalanced**. Plots Precision vs Recall at different thresholds.

In [ ]:
# --- Create Imbalanced Classification Data ---
X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.9, 0.1],  # 90% class 0, 10% class 1
    random_state=42
)

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42
)

print(f"Class distribution (train): {np.bincount(y_train_imb)}")
print(f"Class distribution (test):  {np.bincount(y_test_imb)}")
print(f"Minority class is only {np.mean(y_imb)*100:.1f}% of the data\n")

# --- Train Logistic Regression on imbalanced data ---
log_reg_imb = LogisticRegression(random_state=42, max_iter=1000)
log_reg_imb.fit(X_train_imb, y_train_imb)
y_pred_imb = log_reg_imb.predict(X_test_imb)
y_prob_imb = log_reg_imb.predict_proba(X_test_imb)[:, 1]

# --- Show why accuracy is misleading ---
dummy_acc = np.mean(y_test_imb == 0)  # Always predict majority class
model_acc = accuracy_score(y_test_imb, y_pred_imb)

print("=" * 55)
print("WHY ACCURACY IS MISLEADING WITH IMBALANCED DATA")
print("=" * 55)
print(f"  'Always predict class 0' accuracy: {dummy_acc:.4f}")
print(f"  Logistic Regression accuracy:      {model_acc:.4f}")
print(f"\n  The dummy model looks almost as good by accuracy alone!")
print(f"  But it catches ZERO minority-class samples.\n")

# --- Full classification report ---
print("=" * 55)
print("CLASSIFICATION REPORT (Logistic Regression)")
print("=" * 55)
print(classification_report(y_test_imb, y_pred_imb, target_names=['Class 0 (majority)', 'Class 1 (minority)']))

In [ ]:
# --- Plot Confusion Matrix, ROC Curve, and PR Curve ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test_imb, y_pred_imb)
ConfusionMatrixDisplay(cm, display_labels=['Class 0', 'Class 1']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=13)

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test_imb, y_prob_imb)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, linewidth=2.5, label=f'Logistic Reg (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.500)')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

# 3. Precision-Recall Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test_imb, y_prob_imb)
pr_auc = auc(recall_vals, precision_vals)
axes[2].plot(recall_vals, precision_vals, linewidth=2.5, color='darkorange',
             label=f'Logistic Reg (PR-AUC = {pr_auc:.3f})')
baseline = np.mean(y_test_imb)
axes[2].axhline(y=baseline, color='k', linestyle='--', alpha=0.5,
                label=f'Random (PR-AUC = {baseline:.3f})')
axes[2].set_xlabel('Recall', fontsize=11)
axes[2].set_ylabel('Precision', fontsize=11)
axes[2].set_title('Precision-Recall Curve', fontsize=13)
axes[2].legend(fontsize=10)
axes[2].set_xlim([0, 1])
axes[2].set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

print("Key insight: ROC-AUC can look good even with imbalanced data.")
print("PR-AUC is often more informative for imbalanced datasets.")

---

## Section 3: Decision Trees

### How Decision Trees Work

A decision tree makes predictions by asking a series of **yes/no questions** about the features, creating a flowchart-like structure:

```
                   [Is Feature_1 > 0.5?]
                    /                \
                  Yes                 No
                  /                    \
        [Is Feature_2 > 1.2?]    Predict: Class 0
         /              \
       Yes               No
       /                  \
  Predict: Class 1    Predict: Class 0
```

### How Does the Tree Choose Which Feature to Split On?

The tree tries every possible feature and every possible split point, and picks the one that creates the **purest** groups. Purity is measured by:

**Gini Impurity** (default in sklearn):
$$\text{Gini} = 1 - \sum_{k=1}^{K} p_k^2$$

- Gini = 0 means the node is pure (all samples belong to one class)
- Gini = 0.5 means maximum impurity (50/50 split for binary classification)

**Information Gain** (based on entropy):
$$\text{Entropy} = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

The tree picks the split that reduces impurity the most.

### Pros of Decision Trees
- Very **interpretable** — you can visualize and explain the model
- **No feature scaling** needed (unlike logistic regression)
- Handles **non-linear** relationships naturally
- Can handle both numerical and categorical features

### Cons of Decision Trees
- **Overfitting** — a deep tree will memorize the training data
- **Instability** — small changes in data can create completely different trees
- **High variance** — this is why we use ensembles!

In [ ]:
# --- Fit a Decision Tree and Visualize It ---
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train_2d, y_train_2d)

print(f"Decision Tree (max_depth=3):")
print(f"  Train accuracy: {tree_clf.score(X_train_2d, y_train_2d):.4f}")
print(f"  Test accuracy:  {tree_clf.score(X_test_2d, y_test_2d):.4f}")

# --- Visualize the tree structure ---
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(tree_clf,
          feature_names=['Feature 1', 'Feature 2'],
          class_names=['Class 0', 'Class 1'],
          filled=True,
          rounded=True,
          fontsize=10,
          ax=ax)
ax.set_title('Decision Tree Visualization (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

print("\nEach node shows:")
print("  - The splitting rule (e.g., 'Feature 1 <= 0.42')")
print("  - Gini impurity")
print("  - Number of samples reaching that node")
print("  - Class distribution [class_0_count, class_1_count]")
print("  - Color: blue = more class 0, orange = more class 1")

In [ ]:
# --- Decision Tree Decision Boundary ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_decision_boundary(log_reg, X_test_2d, y_test_2d, ax=axes[0],
                       title='Logistic Regression (Linear Boundary)')
plot_decision_boundary(tree_clf, X_test_2d, y_test_2d, ax=axes[1],
                       title='Decision Tree (Non-Linear Boundary)')

plt.tight_layout()
plt.show()

print("Notice how the decision tree creates RECTANGULAR regions (axis-aligned splits),")
print("while logistic regression creates a single straight line.")

In [ ]:
# --- Demonstrate Overfitting: Train vs Test Accuracy across max_depth ---

depths = range(1, 21)
train_scores = []
test_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train_2d, y_train_2d)
    train_scores.append(tree.score(X_train_2d, y_train_2d))
    test_scores.append(tree.score(X_test_2d, y_test_2d))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(depths, train_scores, 'o-', linewidth=2, markersize=6, label='Train Accuracy', color='steelblue')
ax.plot(depths, test_scores, 's-', linewidth=2, markersize=6, label='Test Accuracy', color='darkorange')
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Decision Tree: Overfitting as Depth Increases', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(list(depths))

# Highlight the best test depth
best_depth = depths[np.argmax(test_scores)]
ax.axvline(x=best_depth, color='green', linestyle='--', alpha=0.7,
           label=f'Best test depth = {best_depth}')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"Best max_depth for test set: {best_depth}")
print(f"  Train accuracy at best depth: {train_scores[best_depth-1]:.4f}")
print(f"  Test accuracy at best depth:  {test_scores[best_depth-1]:.4f}")
print(f"\nAt max_depth=20:")
print(f"  Train accuracy: {train_scores[-1]:.4f} (near-perfect = memorizing!)")
print(f"  Test accuracy:  {test_scores[-1]:.4f} (worse = overfitting!)")

---

## Section 4: Random Forest (Bagging)

### The Idea: Wisdom of the Crowd

A single decision tree is unstable and tends to overfit. What if we trained **many trees** and let them **vote**? That is the core idea behind **Random Forest**.

### How Random Forest Works

1. **Bootstrap sampling**: For each tree, randomly sample N data points **with replacement** from the training set (some points appear multiple times, some not at all)
2. **Random feature selection**: At each split, only consider a **random subset** of features (not all features). This decorrelates the trees.
3. **Build many trees**: Train hundreds of independent decision trees
4. **Aggregate predictions**: For classification, take a **majority vote** across all trees

This process is called **Bagging** (Bootstrap AGGregatING).

### Why Does This Work?

- Each individual tree may overfit to its bootstrap sample
- But the **errors are different** for each tree (because they see different data and features)
- When you average/vote, the individual errors **cancel out**
- Result: lower variance, better generalization

### Key Hyperparameters

| Parameter | What it does | Typical values |
|-----------|-------------|----------------|
| `n_estimators` | Number of trees | 100-1000 (more is usually better, but slower) |
| `max_depth` | Maximum tree depth | None (full), or 5-30 |
| `max_features` | Features considered per split | 'sqrt' (default), 'log2', or a fraction |
| `min_samples_leaf` | Minimum samples in a leaf | 1-10 |
| `oob_score` | Use out-of-bag samples for validation | True/False |

### Out-of-Bag (OOB) Score

Since each tree only sees ~63% of the data (due to bootstrap), the remaining ~37% can be used as a **free validation set**. This is the OOB score — no need for a separate validation split!

In [ ]:
# --- Random Forest: OOB Error vs n_estimators ---

oob_errors = []
n_estimators_range = range(10, 301, 10)

for n_est in n_estimators_range:
    rf = RandomForestClassifier(n_estimators=n_est, oob_score=True, random_state=42, n_jobs=-1)
    rf.fit(X_train_2d, y_train_2d)
    oob_errors.append(1 - rf.oob_score_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(n_estimators_range), oob_errors, linewidth=2, color='steelblue')
ax.set_xlabel('Number of Trees (n_estimators)', fontsize=12)
ax.set_ylabel('OOB Error Rate', fontsize=12)
ax.set_title('Random Forest: OOB Error vs Number of Trees', fontsize=14)
plt.tight_layout()
plt.show()

print("Notice: OOB error decreases and then stabilizes as we add more trees.")
print("After a certain point, adding more trees gives diminishing returns.")

In [ ]:
# --- Random Forest: Feature Importances and Decision Boundary ---

# Train final Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_2d, y_train_2d)

print(f"Random Forest (100 trees):")
print(f"  Train accuracy: {rf_clf.score(X_train_2d, y_train_2d):.4f}")
print(f"  Test accuracy:  {rf_clf.score(X_test_2d, y_test_2d):.4f}")

# --- Feature Importances ---
importances = rf_clf.feature_importances_
feature_names = ['Feature 1', 'Feature 2']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance bar chart
axes[0].barh(feature_names, importances, color=['steelblue', 'darkorange'])
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('Feature Importances (Random Forest)', fontsize=13)
for i, v in enumerate(importances):
    axes[0].text(v + 0.01, i, f'{v:.3f}', fontsize=11, va='center')

# Decision boundary comparison: Single Tree vs Random Forest
plot_decision_boundary(rf_clf, X_test_2d, y_test_2d, ax=axes[1],
                       title='Random Forest — Decision Boundary')

plt.tight_layout()
plt.show()

print("\nCompare: The Random Forest boundary is smoother than a single tree.")
print("This is because it averages many trees, reducing overfitting.")

---

## Section 5: Gradient Boosting

### From Bagging to Boosting

While Random Forest builds trees **independently** (in parallel), **Gradient Boosting** builds trees **sequentially** — each new tree tries to fix the mistakes of the previous ones.

### How Gradient Boosting Works (Step by Step)

1. **Start with a simple prediction** (e.g., the mean for regression, the log-odds for classification)
2. **Calculate the residuals** — the errors of the current model
3. **Fit a new (small) tree to predict the residuals** — this tree learns the "mistakes"
4. **Add this tree to the ensemble** (scaled by a learning rate)
5. **Repeat steps 2-4** for `n_estimators` rounds

Each round reduces the error a little bit, gradually building a strong model from many weak learners.

### The Learning Rate / n_estimators Tradeoff

- **Learning rate** (shrinkage): Controls how much each tree contributes. Smaller = more conservative.
- **n_estimators**: Number of boosting rounds.
- **Key tradeoff**: A smaller learning rate needs more trees (but often gives better results)
  - `learning_rate=0.1, n_estimators=100` might perform similarly to `learning_rate=0.01, n_estimators=1000`
  - The second option is slower but often generalizes better

### XGBoost and LightGBM

These are **optimized implementations** of gradient boosting that are faster and often more accurate:

| Library | Key Advantages |
|---------|---------------|
| **XGBoost** | Regularization built-in, handles missing values, parallel processing, very mature |
| **LightGBM** | Faster training (leaf-wise growth), lower memory, handles large datasets, categorical features natively |

Both are industry standards for tabular data and dominate Kaggle competitions.

### Key Hyperparameters

| Parameter | What it does | Typical values |
|-----------|-------------|----------------|
| `n_estimators` | Number of boosting rounds | 100-3000 |
| `learning_rate` | Step size shrinkage | 0.01-0.3 |
| `max_depth` | Tree depth (keep shallow!) | 3-8 |
| `subsample` | Fraction of data per tree | 0.7-1.0 |
| `colsample_bytree` | Fraction of features per tree | 0.7-1.0 |

In [ ]:
# --- Import XGBoost and LightGBM (optional) ---
try:
    from xgboost import XGBClassifier
    has_xgb = True
    print("XGBoost imported successfully!")
except ImportError:
    has_xgb = False
    print("XGBoost not available. Install with: pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    has_lgbm = True
    print("LightGBM imported successfully!")
except ImportError:
    has_lgbm = False
    print("LightGBM not available. Install with: pip install lightgbm")

In [ ]:
# --- Generate a more complex dataset for boosting comparison ---
import time

X_complex, y_complex = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_clusters_per_class=2,
    random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_complex, y_complex, test_size=0.3, random_state=42
)

print(f"Complex dataset: {X_complex.shape[0]} samples, {X_complex.shape[1]} features")
print(f"Train: {X_train_c.shape[0]}, Test: {X_test_c.shape[0]}")

# --- Compare models ---
models = {
    'sklearn GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42
    ),
}

if has_xgb:
    models['XGBoost'] = XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=42, eval_metric='logloss', verbosity=0
    )

if has_lgbm:
    models['LightGBM'] = LGBMClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=42, verbose=-1
    )

results = []

for name, model in models.items():
    start_time = time.time()
    model.fit(X_train_c, y_train_c)
    train_time = time.time() - start_time

    train_acc = model.score(X_train_c, y_train_c)
    test_acc = model.score(X_test_c, y_test_c)

    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Train Time (s)': train_time
    })

    print(f"\n{name}:")
    print(f"  Train accuracy: {train_acc:.4f}")
    print(f"  Test accuracy:  {test_acc:.4f}")
    print(f"  Train time:     {train_time:.3f}s")

results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print(results_df.to_string(index=False))

In [ ]:
# --- Learning Curves: Score vs n_estimators for Gradient Boosting ---

# We'll use staged_predict from sklearn's GradientBoosting to get score at each stage
gb_clf = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42
)
gb_clf.fit(X_train_c, y_train_c)

# Get staged predictions (score after each tree is added)
train_staged_scores = []
test_staged_scores = []

for y_pred_staged in gb_clf.staged_predict(X_train_c):
    train_staged_scores.append(accuracy_score(y_train_c, y_pred_staged))

for y_pred_staged in gb_clf.staged_predict(X_test_c):
    test_staged_scores.append(accuracy_score(y_test_c, y_pred_staged))

fig, ax = plt.subplots(figsize=(10, 6))
n_trees = range(1, len(train_staged_scores) + 1)
ax.plot(n_trees, train_staged_scores, linewidth=2, label='Train Accuracy', color='steelblue')
ax.plot(n_trees, test_staged_scores, linewidth=2, label='Test Accuracy', color='darkorange')
ax.set_xlabel('Number of Boosting Rounds (n_estimators)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Gradient Boosting: Learning Curve', fontsize=14)
ax.legend(fontsize=11)

# Find best test score
best_n = np.argmax(test_staged_scores) + 1
ax.axvline(x=best_n, color='green', linestyle='--', alpha=0.7,
           label=f'Best test score at n={best_n}')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"Best test accuracy: {max(test_staged_scores):.4f} at {best_n} trees")
print("Notice: train accuracy keeps increasing, but test accuracy plateaus or drops.")
print("This is why early stopping is important for gradient boosting!")

---

## Section 6: Hyperparameter Tuning

### Grid Search vs Random Search

When you have a model with multiple hyperparameters, you need to find the best combination. Two common approaches:

**Grid Search** (`GridSearchCV`):
- Tries **every combination** of parameter values you specify
- Exhaustive but can be slow with many parameters
- Best when: few hyperparameters, small search space

**Random Search** (`RandomizedSearchCV`):
- Samples random combinations from the parameter space
- More efficient — often finds good results faster
- Best when: many hyperparameters, large search space

### Important Hyperparameters by Model

| Model | Key Parameters to Tune |
|-------|----------------------|
| **Logistic Regression** | `C` (regularization strength), `penalty` (L1/L2) |
| **Decision Tree** | `max_depth`, `min_samples_split`, `min_samples_leaf` |
| **Random Forest** | `n_estimators`, `max_depth`, `max_features`, `min_samples_leaf` |
| **Gradient Boosting** | `n_estimators`, `learning_rate`, `max_depth`, `subsample` |

### Cross-Validation in Grid Search

`GridSearchCV` automatically uses **k-fold cross-validation** to evaluate each parameter combination. This gives a more reliable estimate than a single train/test split.

In [ ]:
# --- GridSearchCV on Random Forest ---

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 3, 5]
}

print("Parameter grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

total_combos = 1
for v in param_grid.values():
    total_combos *= len(v)
print(f"\nTotal combinations to try: {total_combos}")
print(f"With 5-fold CV, that's {total_combos * 5} model fits!\n")

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True,
    n_jobs=-1
)

grid_search.fit(X_train_c, y_train_c)

print("=" * 50)
print("GRID SEARCH RESULTS")
print("=" * 50)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score:   {grid_search.best_score_:.4f}")
print(f"Test score:      {grid_search.score(X_test_c, y_test_c):.4f}")

In [ ]:
# --- Visualize GridSearch Results as a Heatmap ---

# Extract results into a DataFrame
cv_results = pd.DataFrame(grid_search.cv_results_)

# Create a pivot table for heatmap: max_depth vs n_estimators (averaging over min_samples_leaf)
# We'll fix min_samples_leaf to the best value for a cleaner heatmap
best_msl = grid_search.best_params_['min_samples_leaf']

# Filter to best min_samples_leaf
mask = cv_results['param_min_samples_leaf'] == best_msl
filtered = cv_results[mask]

pivot = filtered.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators'
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu', ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title(f'CV Accuracy: max_depth vs n_estimators\n(min_samples_leaf={best_msl})', fontsize=13)
ax.set_xlabel('n_estimators', fontsize=12)
ax.set_ylabel('max_depth', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Heatmap shows CV accuracy for each (max_depth, n_estimators) combination")
print(f"with min_samples_leaf fixed at {best_msl} (the best value from grid search).")

---

## Section 7: Model Comparison

Let's put all models head-to-head on the same dataset using cross-validation for a fair comparison.

In [ ]:
# --- Compare All Models with Cross-Validation ---

comparison_models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                     max_depth=5, random_state=42),
}

if has_xgb:
    comparison_models['XGBoost'] = XGBClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=5,
        random_state=42, eval_metric='logloss', verbosity=0
    )

if has_lgbm:
    comparison_models['LightGBM'] = LGBMClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=5,
        random_state=42, verbose=-1
    )

# Use the complex dataset
cv_results_all = {}
summary_rows = []

print("Running 5-fold cross-validation for each model...\n")
print(f"{'Model':<25} {'Mean CV Score':>14} {'Std':>8} {'Train Time':>12}")
print("-" * 62)

for name, model in comparison_models.items():
    start_time = time.time()
    scores = cross_val_score(model, X_complex, y_complex, cv=5, scoring='accuracy', n_jobs=-1)
    elapsed = time.time() - start_time

    cv_results_all[name] = scores
    summary_rows.append({
        'Model': name,
        'Mean CV Score': scores.mean(),
        'Std': scores.std(),
        'Train Time (s)': elapsed
    })

    print(f"{name:<25} {scores.mean():>14.4f} {scores.std():>8.4f} {elapsed:>11.3f}s")

print("\n" + "=" * 62)
summary_df = pd.DataFrame(summary_rows).sort_values('Mean CV Score', ascending=False)
print("\nRanked by Mean CV Score:")
print(summary_df.to_string(index=False))

In [ ]:
# --- Box Plot of CV Scores ---

fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(cv_results_all.keys())
scores_list = [cv_results_all[name] for name in model_names]

bp = ax.boxplot(scores_list, labels=model_names, patch_artist=True, widths=0.6)

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860']
for patch, color in zip(bp['boxes'], colors[:len(model_names)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('CV Accuracy', fontsize=12)
ax.set_title('Model Comparison: 5-Fold Cross-Validation Scores', fontsize=14)
ax.tick_params(axis='x', rotation=20)

# Add mean markers
means = [scores.mean() for scores in scores_list]
ax.scatter(range(1, len(model_names) + 1), means, color='red', marker='D',
           s=60, zorder=5, label='Mean')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---

## Key Takeaways

### Model Summary

| Model | Strengths | Weaknesses | When to Use |
|-------|-----------|------------|-------------|
| **Logistic Regression** | Fast, interpretable, strong baseline, probabilistic output | Only linear boundaries | Always start here as a baseline |
| **Decision Tree** | Interpretable, no scaling needed, handles non-linear | Overfits easily, unstable | When you need explainability |
| **Random Forest** | Robust, hard to mess up, reduces overfitting | Slower, less interpretable than single tree | Reliable default for most problems |
| **XGBoost / LightGBM** | Usually best performance, handles complex patterns | More hyperparameters to tune, less interpretable | When you want maximum performance |

### Decision Flowchart for Classification

```
Start with your classification problem
    |
    v
1. Train Logistic Regression (baseline)
    |
    v
2. Train Random Forest (usually better, still easy)
    |
    v
3. Train XGBoost or LightGBM (best performance, needs tuning)
    |
    v
4. Tune the best model with GridSearchCV or RandomizedSearchCV
    |
    v
5. Evaluate with appropriate metrics (F1, ROC-AUC, PR-AUC)
   - Use F1 or PR-AUC for imbalanced data
   - Use ROC-AUC for balanced data
```

### Key Lessons

- **Never trust accuracy alone** — especially with imbalanced data. Always look at precision, recall, F1, and the confusion matrix.
- **Logistic Regression is a strong baseline** — it is fast, interpretable, and often surprisingly competitive.
- **Decision trees overfit** — but they are the building blocks for powerful ensemble methods.
- **Random Forest is reliable** — it is hard to get wrong and rarely overfits badly. A good "set and forget" model.
- **Gradient Boosting (XGBoost/LightGBM) often wins** — but requires more careful tuning (learning rate, n_estimators, max_depth).
- **Always use cross-validation** — a single train/test split can be misleading.

---

## Exercises

### Exercise 1: Compare Models on Breast Cancer Dataset

Load sklearn's `breast_cancer` dataset. Train Logistic Regression, Random Forest, and GradientBoosting. Compare them using **5-fold cross-validated F1 scores** (use `scoring='f1'`). Which model performs best?

In [ ]:
# Exercise 1: Compare models on breast cancer dataset
# -----------------------------------------------
from sklearn.datasets import load_breast_cancer

# Load the data
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
print(f"Breast Cancer dataset: {X_bc.shape[0]} samples, {X_bc.shape[1]} features")
print(f"Classes: {data.target_names}")
print(f"Class distribution: {np.bincount(y_bc)}")

# YOUR CODE HERE:
# 1. Create a dictionary of models (LogisticRegression, RandomForest, GradientBoosting)
# 2. For each model, run cross_val_score with cv=5 and scoring='f1'
# 3. Print the mean and std of F1 scores for each model
# 4. Which model wins?

# models_ex1 = {
#     'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
#     'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
#     'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
# }
#
# for name, model in models_ex1.items():
#     scores = cross_val_score(model, X_bc, y_bc, cv=5, scoring='f1')
#     print(f"{name:<25} F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

### Exercise 2: Precision-Recall Tradeoff

Using the imbalanced dataset from Section 2, vary the classification threshold from 0.1 to 0.9 and plot how **precision** and **recall** change. At what threshold is the F1-score maximized?

In [ ]:
# Exercise 2: Precision-Recall Tradeoff
# -----------------------------------------------
# We already have y_prob_imb (predicted probabilities from the imbalanced dataset)
# and y_test_imb (true labels)

# YOUR CODE HERE:
# 1. Create a list of thresholds from 0.1 to 0.9 (e.g., np.arange(0.1, 0.95, 0.05))
# 2. For each threshold, convert probabilities to predictions: (y_prob_imb >= threshold).astype(int)
# 3. Calculate precision, recall, and F1 for each threshold
# 4. Plot precision and recall vs threshold on the same plot
# 5. Find the threshold that maximizes F1

# thresholds = np.arange(0.1, 0.95, 0.05)
# precisions = []
# recalls = []
# f1s = []
#
# for t in thresholds:
#     y_pred_t = (y_prob_imb >= t).astype(int)
#     precisions.append(precision_score(y_test_imb, y_pred_t, zero_division=0))
#     recalls.append(recall_score(y_test_imb, y_pred_t, zero_division=0))
#     f1s.append(f1_score(y_test_imb, y_pred_t, zero_division=0))
#
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.plot(thresholds, precisions, 'o-', label='Precision', linewidth=2)
# ax.plot(thresholds, recalls, 's-', label='Recall', linewidth=2)
# ax.plot(thresholds, f1s, '^-', label='F1-Score', linewidth=2)
# ax.set_xlabel('Classification Threshold', fontsize=12)
# ax.set_ylabel('Score', fontsize=12)
# ax.set_title('Precision-Recall Tradeoff', fontsize=14)
# ax.legend(fontsize=11)
# best_threshold = thresholds[np.argmax(f1s)]
# ax.axvline(x=best_threshold, color='green', linestyle='--', alpha=0.7,
#            label=f'Best F1 threshold = {best_threshold:.2f}')
# ax.legend(fontsize=11)
# plt.tight_layout()
# plt.show()
# print(f"Best F1-score: {max(f1s):.4f} at threshold {best_threshold:.2f}")

### Exercise 3: Hyperparameter Tuning with GridSearchCV

Use `GridSearchCV` to tune a `RandomForestClassifier` on the breast cancer dataset. Search over:
- `max_depth`: [3, 5, 10, None]
- `n_estimators`: [50, 100, 200]
- `min_samples_leaf`: [1, 2, 5]

Use 5-fold CV with `scoring='f1'`. Report the best parameters and best score.

In [ ]:
# Exercise 3: Hyperparameter Tuning with GridSearchCV
# -----------------------------------------------
# Use the breast cancer dataset (X_bc, y_bc) from Exercise 1

# YOUR CODE HERE:
# 1. Define the parameter grid
# 2. Create a GridSearchCV object with RandomForestClassifier, cv=5, scoring='f1'
# 3. Fit on the data
# 4. Print best_params_ and best_score_
# 5. (Bonus) Evaluate the best model on a held-out test set

# param_grid_ex3 = {
#     'max_depth': [3, 5, 10, None],
#     'n_estimators': [50, 100, 200],
#     'min_samples_leaf': [1, 2, 5],
# }
#
# grid_ex3 = GridSearchCV(
#     RandomForestClassifier(random_state=42, n_jobs=-1),
#     param_grid_ex3,
#     cv=5,
#     scoring='f1',
#     n_jobs=-1
# )
#
# grid_ex3.fit(X_bc, y_bc)
#
# print(f"Best parameters: {grid_ex3.best_params_}")
# print(f"Best F1 score:   {grid_ex3.best_score_:.4f}")